# Objetivo do Projeto

O objetivo deste projeto é desenvolver um **modelo preditivo capaz de estimar a probabilidade de inadimplência de cobranças mensais realizadas aos clientes**.

A inadimplência é definida como um pagamento realizado com **5 dias ou mais de atraso em relação à data de vencimento**.

O modelo deve gerar uma probabilidade de inadimplência para cada registro presente na base de pagamentos, que representa as cobranças mais recentes da "empresa".

Para o desenvolvimento do modelo, estão disponíveis diferentes fontes de informação sobre os clientes:

* **Histórico de pagamentos**, contendo informações sobre o comportamento de pagamento;
* **Dados cadastrais e de perfil** dos clientes;
* **Informações mensais**, como renda e número de funcionários;
* Outras informações históricas que podem auxiliar na identificação de padrões associados à inadimplência.

A solução deve considerar aspectos como:

* tratamento e qualidade dos dados;
* criação e seleção de variáveis;
* separação adequada entre treino, validação e teste;
* avaliação do desempenho do modelo;
* interpretação dos resultados;
* comparação com uma regra de decisão já existente (*proxy*).

A saída final deve conter **apenas a probabilidade de inadimplência para cada cobrança**, com valores entre `0` e `1`.

Não é necessário transformar essas probabilidades em uma classificação binária de inadimplente ou adimplente.

> **Observação:** a definição de inadimplência utilizada neste projeto — **5 dias ou mais de atraso** — é adotada como regra para construção do alvo e avaliação do modelo.


# Base de Dados


O projeto disponibiliza **três bases de dados** contendo informações cadastrais, comportamentais e financeiras dos clientes. As bases foram extraídas de um sistema de cobrança e representam um cenário realista de operação.

As tabelas se relacionam principalmente por meio de:

* **ID_CLIENTE**: identifica unicamente cada cliente;
* **SAFRA_REF**: representa o período de referência da cobrança.

Bases disponíveis

| Base                                  | Descrição                                                                       | Granularidade      |
| ------------------------------------- | ------------------------------------------------------------------------------- | ------------------ |
| cadastral                  | Informações cadastrais, como data de cadastro, porte, CEP e domínio do e-mail.  | Cliente            |
| info                      | Informações mensais, como renda e número de funcionários.                       | Cliente × mês      |
| pagamentos          | Cobranças mais recentes, utilizadas para gerar  treinamento e previsões do modelo.          | Cliente × cobrança |

### Papel de cada base

A **cadastral** contém informações mais estáticas sobre os clientes, enquanto a **info** permite acompanhar características que podem variar ao longo do tempo.

A **pagamentos** é utilizada para o desenvolvimento do modelo. Como possui as datas de vencimento e pagamento, é possível identificar quais cobranças foram inadimplentes de acordo com a regra definida no projeto.


# Cadastral


A base **cadastral** reúne informações de identificação e características cadastrais dos clientes. Cada registro representa um cliente único, identificado por ID_CLIENTE.

A base possui **1.315 registros e 8 variáveis**.

Durante a exploração inicial, foram observados alguns pontos de atenção, principalmente a presença de valores ausentes e inconsistências na variável DDD, que apresenta registros contendo números e caracteres.

Dicionário de Dados

| Variável              | Descrição                             | Tipo                | Observações                                     |
| --------------------- | ------------------------------------- | ------------------- | ----------------------------------------------- |
| ID_CLIENTE            | Identificador único do cliente        | Inteiro             | Chave da base                                   |
| DATA_CADASTRO         | Data em que o cliente foi cadastrado  | Data                | Sem valores nulos                               |
| DDD                   | Código de área do telefone do cliente | Texto/Numérico      | ~18% nulos e presença de valores inconsistentes |
| FLAG_PF               | Indicador de pessoa física            | Binário             | ~95% nulos                                      |
| SEGMENTO_INDUSTRIAL   | Segmento de atuação do cliente        | Categórico          | ~6,3% nulos                                     |
|  DOMINIO_EMAIL        | Domínio do e-mail cadastrado          | Categórico          | ~2,3% nulos                                     |
|  PORTE                | Porte do cliente/empresa              | Categórico          | ~3,1% nulos                                     |
|  CEP_2_DIG            | Dois primeiros dígitos do CEP         | Numérico/Categórico | ~0,08% nulos                                    |

Pontos observados

* FLAG_PF apresenta uma **alta concentração de valores nulos**, sendo consideravel descarte.
* DDD apresenta **inconsistências de formato**, com valores contendo números e letras, além de uma quantidade relevante de nulos.
* As demais variáveis apresentam níveis de ausência menores e poderão ser tratadas durante a etapa de preparação dos dados.
* ID_CLIENTE será utilizado como chave para relacionar esta base às demais fontes de dados.


In [0]:
cadastral_df = spark.table("credit_score.data.cadastral")
display(cadastral_df.limit(10))


In [0]:
num_rows = cadastral_df.count()
num_cols = len(cadastral_df.columns)
display(spark.createDataFrame([(num_rows, num_cols)], ["num_linhas", "num_colunas"]))

In [0]:
display(cadastral_df.dtypes)

In [0]:
display(cadastral_df.select("DDD").distinct())

In [0]:
display(cadastral_df.select("CEP_2_DIG").distinct())

In [0]:
from pyspark.sql.functions import col, count, when

total_count = cadastral_df.count()
missing_pct_df = (
    cadastral_df.select([
        (count(when(col(c).isNull(), c)) / total_count * 100).alias(c)
        for c in cadastral_df.columns
    ])
)
display(missing_pct_df)

# Info


A base **info** contém informações mensais relacionadas aos clientes, permitindo acompanhar características que podem variar ao longo do tempo. Cada registro representa um cliente em determinado período de referência (SAFRA_REF).

A base possui **24.401 registros e 4 variáveis**.

Dicionário de Dados

| Variável             | Descrição                                                 | Tipo     | Observações                              |
| -------------------- | --------------------------------------------------------- | -------- | ---------------------------------------- |
| ID_CLIENTE         | Identificador único do cliente                            | Inteiro  | Chave para relacionamento entre as bases |
| SAFRA_REF          | Período de referência das informações                     | Data     | Representa o mês de referência           |
| RENDA_MES_ANTERIOR | Renda registrada no mês anterior ao período de referência | Numérico | ~2,94% nulos                             |
| NO_FUNCIONARIOS    | Número de funcionários do cliente                         | Numérico | ~5,13% nulos                             |

Pontos observados

* A base possui uma **estrutura temporal**, permitindo analisar a evolução das características dos clientes ao longo dos meses.
* RENDA_MES_ANTERIOR apresenta aproximadamente **2,94% de valores ausentes**.
* NO_FUNCIONARIOS possui aproximadamente **5,13% de valores ausentes**.
* ID_CLIENTE e SAFRA_REF não apresentam valores nulos e serão importantes para o relacionamento temporal com as demais bases.
* Como essas informações variam ao longo do tempo, é importante garantir que, na construção das variáveis, sejam utilizadas apenas informações **disponíveis até o período de referência**, evitando vazamento de informação (*data leakage*).


In [0]:
info_df = spark.table("credit_score.data.info")
display(info_df.limit(10))


In [0]:
num_rows = info_df.count()
num_cols = len(info_df.columns)
display(spark.createDataFrame([(num_rows, num_cols)], ["num_linhas", "num_colunas"]))

In [0]:
display(info_df.dtypes)

In [0]:
from pyspark.sql.functions import col, count, when

total_count = info_df.count()
missing_pct_df = (
    info_df.select([
        (count(when(col(c).isNull(), c)) / total_count * 100).alias(c)
        for c in info_df.columns
    ])
)
display(missing_pct_df)

# Pagamentos


A base **pagamentos** contém o histórico de cobranças e pagamentos realizados pelos clientes. Cada registro representa uma cobrança associada a um cliente e a um período de referência.

Essa é uma das principais bases para o projeto, pois contém as **datas de vencimento e pagamento**, permitindo identificar a ocorrência de inadimplência de acordo com a regra definida.

A base possui **77.414 registros e 8 variáveis**.

Dicionário de Dados

| Variável                 | Descrição                                 | Tipo     | Observações                       |
| ------------------------ | ----------------------------------------- | -------- | --------------------------------- |
| ID_CLIENTE             | Identificador do cliente                  | Inteiro  | Chave de relacionamento           |
| SAFRA_REF              | Período de referência da cobrança         | Data     | Permite o relacionamento temporal |
| DATA_EMISSAO_DOCUMENTO | Data de emissão da cobrança               | Data     | Sem valores nulos                 |
| DATA_PAGAMENTO         | Data em que o pagamento foi realizado     | Data     | Sem valores nulos                 |
| DATA_VENCIMENTO        | Data de vencimento da cobrança            | Data     | Sem valores nulos                 |
| VALOR_A_PAGAR          | Valor da cobrança                         | Numérico | ~1,51% nulos                      |
| TAXA                   | Taxa associada à cobrança                 | Numérico | Sem valores nulos                 |
| ID_DOCUMENTO           | Identificador único do documento/cobrança | Inteiro  | Chave do documento                |


Pontos observados

* A base possui as informações necessárias para **construção da variável target**, comparando DATA_PAGAMENTO e DATA_VENCIMENTO.
* A regra utilizada no projeto considera uma cobrança **inadimplente quando o pagamento ocorre com 5 dias ou mais de atraso**.
* VALOR_A_PAGAR apresenta aproximadamente **1,51% de valores ausentes**, sendo necessário tratar esses registros durante o desenvolvimento.
* ID_DOCUMENTO permite identificar individualmente cada cobrança e também é utilizado no relacionamento com outras informações do projeto.
* As datas disponíveis permitem construir variáveis relacionadas ao **comportamento de pagamento e histórico de atrasos**.


In [0]:
pagamentos_df = spark.table("credit_score.data.pagamentos")
display(pagamentos_df.limit(10))


In [0]:
num_rows = pagamentos_df.count()
num_cols = len(pagamentos_df.columns)
display(spark.createDataFrame([(num_rows, num_cols)], ["num_linhas", "num_colunas"]))

In [0]:
display(pagamentos_df.dtypes)

In [0]:
from pyspark.sql.functions import col, count, when

total_count = pagamentos_df.count()
missing_pct_df = (
    pagamentos_df.select([
        (count(when(col(c).isNull(), c)) / total_count * 100).alias(c)
        for c in pagamentos_df.columns
    ])
)
display(missing_pct_df)